In [ ]:
%load_ext rich

In [ ]:
import json
import os
from pathlib import Path

import requests
from tqdm import tqdm

API_URL = "http://localhost:8000"  # Url for debugger. change it to your own
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}

## Sample document


In [ ]:
def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )

In [ ]:
documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)

print(f"Found {len(documents)} documents")

doc_path = documents[16]

## /document-extract endpoint output


In [ ]:
import mimetypes
import time
from pathlib import Path
from typing import Any, Iterable

from more_itertools import unique_everseen


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    """
    Discover documents in a directory with given extensions.

    Args:
        root (Path): Root directory to search for documents.
        extensions (Iterable[str]): File extensions to include.

    Returns:
        list[Path]: List of discovered document paths with the specified extensions.
    """
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


def call_extraction_api(
    session: requests.Session, endpoint: str, file_path: Path, timeout_s: float
) -> dict[str, object]:
    """
    Call the extraction API with a document file.

    Args:
        session (requests.Session): HTTP session for making requests.
        endpoint (str): URL of the extraction API endpoint.
        file_path (Path): Path to the document file to be processed.
        timeout_s (float): Request timeout in seconds.

    Returns:
        dict[str, object]: Payload containing the response details.
    """
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            endpoint,
            files=files,
            timeout=timeout_s,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload


def parse_prediction_labels(predictions: list[dict[str, Any]]) -> list[dict[str, str]]:
    """
    Parse prediction labels to extract unique aymurai labels and their alternative texts.

    Args:
        predictions (list[dict[str, Any]]): List of prediction dictionaries.

    Returns:
        list[dict[str, str]]: List of dictionaries containing unique aymurai labels and their alternative texts.
    """
    attrs_stream = (
        label.get("attrs") or {}
        for label in (label for pred in predictions for label in pred.get("labels", ()))
    )

    unique_pairs = unique_everseen(
        (
            attrs.get("aymurai_label"),
            attrs.get("aymurai_alt_text"),
        )
        for attrs in attrs_stream
        if attrs.get("aymurai_label") and attrs.get("aymurai_alt_text")
    )

    return sorted(
        ({"aymurai_label": label, "text": text} for label, text in unique_pairs),
        key=lambda item: (item["aymurai_label"], item["text"]),
    )

In [ ]:
# /document-extract endpoint output
session = requests.Session()
document = call_extraction_api(
    session,
    endpoint=f"{API_URL}/misc/document-extract",
    file_path=doc_path,
    timeout_s=300,
)
paragraphs = document["detail"]["document"]
document

In [ ]:
len(paragraphs)

## Inference


In [ ]:
# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample}, params={"use_cache": False})
    response.raise_for_status()
    return response.json()

In [ ]:
predictions = [get_predictions(paragraph) for paragraph in tqdm(paragraphs)]
predictions

## Export variants


In [ ]:
def disambiguate_and_export(
    variant_name: str, label_policies: dict, render_policy: dict
):
    response = requests.post(
        url=f"{API_URL}/anonymizer/disambiguate",
        json={
            "paragraphs": predictions,
            # "custom_prompts": {"root": []},
            "label_policies": label_policies,
        },
    )
    response.raise_for_status()
    disambiguated = response.json()

    json_prediction = json.dumps(
        {
            "data": disambiguated["data"],
            "label_policies": label_policies,
            "render_policy": render_policy,
        }
    )

    with open(doc_path, "rb") as file:
        files = {"file": file}

        response = requests.post(
            url=f"{API_URL}/anonymizer/anonymize-document",
            data={"annotations": json_prediction},
            files=files,
        )
        response.raise_for_status()

    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    filename = os.path.basename(doc_path)
    filename, ext = os.path.splitext(filename)
    out_path = f"{output_dir}/{filename}-{variant_name}.odt"

    with open(out_path, "wb") as file:
        file.write(response.content)

    return out_path

In [ ]:
render_policy = {
    "suffix_mode": "auto",
    "suffix_threshold": 1,
}

# 1) everything fuzzy
label_policies_all_fuzzy = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
}
out_all_fuzzy = disambiguate_and_export(
    "all-fuzzy", label_policies_all_fuzzy, render_policy
)

# 2) FECHAs excluded
label_policies_no_fecha = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "none", "anonymize": False, "use_subclass_when_available": False},
}
out_no_fecha = disambiguate_and_export(
    "no-fecha", label_policies_no_fecha, render_policy
)

# 3) PER llm-disambiguated
label_policies_per_llm = {
    "PER": {"disambiguation": "llm", "anonymize": False, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "none", "anonymize": True, "use_subclass_when_available": False},
}
out_per_llm = disambiguate_and_export("per-llm", label_policies_per_llm, render_policy)

out_all_fuzzy, out_no_fecha, out_per_llm